# تدريب Qwen2.5-3B-Instruct على القوانين اليمنية باستخدام MoRA

هذا الدفتر ينفذ التدريب من الفرع الخاص `moranew` باستخدام QLoRA بتحميل 4-bit وMoRA. ملف البيانات يُنزّل من رابط Google Drive المعتمد، ثم يُفحص قبل التدريب. الناتج adapter قانوني مصدر-مقيد، وليس بديلاً عن محامٍ مرخّص أو المراجعة البشرية.

In [ ]:
# 1) فحص GPU
!nvidia-smi
import sys, os, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## 2. جلب الفرع الخاص

أضف في Colab Secret مفتاحاً باسم `GITHUB_TOKEN` يملك صلاحية قراءة المستودع الخاص، أو ارفع المشروع يدوياً إلى `/content/qwen25-3b-yemeni-mora`. لا تكتب المفتاح داخل الدفتر ولا تطبعه في المخرجات.

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/EngKHALIDx/qwen25-3b-yemeni-mora.git"
REPO_BRANCH = "moranew"
REPO_DIR = Path("/content/qwen25-3b-yemeni-mora")

if not REPO_DIR.exists():
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None
    if not github_token:
        raise RuntimeError(
            "المستودع خاص. أضف Colab Secret باسم GITHUB_TOKEN أو ارفع مجلد المشروع إلى /content/qwen25-3b-yemeni-mora."
        )
    authenticated_url = REPO_URL.replace(
        "https://", f"https://x-access-token:{github_token}@", 1
    )
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", authenticated_url, str(REPO_DIR)],
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )

assert (REPO_DIR / "train.py").exists(), "لم يتم العثور على train.py داخل المشروع"
os.chdir(REPO_DIR)
print("Project:", REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())

In [ ]:
# 3) تثبيت الاعتماديات. Colab يوفر PyTorch CUDA؛ لا نعيد تثبيته من requirements.
!pip install -q -r requirements.txt
!python -m pip show torch transformers accelerate bitsandbytes | grep -E '^(Name|Version):' || true

## 4. ربط Google Drive وحفظ النتائج

يُحفظ adapter وcheckpoints في Google Drive حتى يمكن استئناف التدريب بعد انتهاء جلسة Colab.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/qwen25_3b_mora_adapter")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

## 5. تنزيل ملف البيانات من الرابط المعتمد

الملف غير مضغوط، وحجمه يقارب 596 MiB. سيُحفظ مؤقتاً في مساحة Colab، ثم يتحقق الدفتر من SHA-256 قبل التدريب.

In [ ]:
import hashlib, json, requests
from pathlib import Path
from tqdm.auto import tqdm

DRIVE_FILE_ID = "1U9-DSU0_GH4LXu1SmmPeKoK7dhxwrFPz"
DRIVE_URL = f"https://drive.usercontent.google.com/download?id={DRIVE_FILE_ID}&export=download&confirm=t"
EXPECTED_CONTENT_SHA256 = "bbd4721b20b02c2d5a97b594c530354548a2dcc01b5c8f508539abe247400dfe"
DATA_DIR = Path("/content/qwen25_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "qwen25_3b_semantic_colab.jsonl"

if not DATA_PATH.exists() or DATA_PATH.stat().st_size < 100_000_000:
    with requests.get(DRIVE_URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with DATA_PATH.open("wb") as handle, tqdm(total=total, unit="B", unit_scale=True, desc="Downloading") as bar:
            for block in response.iter_content(chunk_size=1024 * 1024):
                if block:
                    handle.write(block)
                    bar.update(len(block))
print("Data path:", DATA_PATH)
print("Bytes:", DATA_PATH.stat().st_size)

In [ ]:
# 6) تحقق كامل تدفقياً من البصمة والبنية وعدد السجلات
import gzip, hashlib, json

count = 0
malformed = 0
no_assistant = 0
max_token_count = 0
digest = hashlib.sha256()
with DATA_PATH.open("rb") as raw:
    for block in iter(lambda: raw.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_CONTENT_SHA256, digest.hexdigest()

with DATA_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            malformed += 1
            continue
        count += 1
        messages = record.get("messages")
        if not isinstance(messages, list) or {m.get("role") for m in messages} != {"system", "user", "assistant"}:
            malformed += 1
        if not any(m.get("role") == "assistant" and m.get("content", "").strip() for m in messages if isinstance(m, dict)):
            no_assistant += 1
        max_token_count = max(max_token_count, int(record.get("token_count_qwen25_chat", 0)))

assert count == 165303, count
assert malformed == 0, malformed
assert no_assistant == 0, no_assistant
assert max_token_count <= 1024, max_token_count
print({"status": "PASS", "records": count, "max_token_count_qwen25_chat": max_token_count, "sha256": digest.hexdigest()})

## 7. اختبار دخان قصير

لا تبدأ التدريب الكامل قبل نجاح هذه الخلية. الاختبار يستخدم 128 سجلاً و5 خطوات فقط، ويتأكد من تحميل Qwen، واستيراد MoRA المحلي، وتطبيق قالب المحادثة، وتمرير batch على GPU.

In [ ]:
!python train.py \
  --config configs/qwen25_3b_mora.json \
  --data_path /content/qwen25_data/qwen25_3b_semantic_colab.jsonl \
  --max_train_samples 128 \
  --max_steps 5 \
  --output_dir /content/qwen25_mora_smoke_test

## 8. التدريب الكامل

الإعداد الافتراضي مناسب لبداية T4/V100: QLoRA 4-bit، batch فعلي 1، gradient accumulation يساوي 16، وgradient checkpointing مفعّل. إذا انقطعت الجلسة، شغّل خلية الاستئناف التالية بدلاً من هذه الخلية.

In [ ]:
!python train.py \
  --config configs/qwen25_3b_mora.json \
  --data_path /content/qwen25_data/qwen25_3b_semantic_colab.jsonl \
  --output_dir /content/drive/MyDrive/qwen25_3b_mora_adapter

## 9. استئناف التدريب من آخر checkpoint

استخدم هذه الخلية فقط إذا توقف التدريب بعد إنشاء checkpoint صالح. غيّر المسار إلى اسم آخر مجلد `checkpoint-*` موجود.

In [ ]:
# مثال، لا تشغّلها إلا بعد تعديل المسار:
# !python train.py \\
#   --config configs/qwen25_3b_mora.json \\
#   --data_path /content/qwen25_data/qwen25_3b_semantic_colab.jsonl \\
#   --output_dir /content/drive/MyDrive/qwen25_3b_mora_adapter \\
#   --resume_from_checkpoint /content/drive/MyDrive/qwen25_3b_mora_adapter/checkpoint-500

## 10. اختبار adapter بعد التدريب

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

adapter_dir = str(OUTPUT_DIR)
base_id = "Qwen/Qwen2.5-3B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
base = AutoModelForCausalLM.from_pretrained(
    base_id,
    quantization_config=bnb,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

# استخدام سؤال من سجل rule مع حذف إجابة المرجع حتى يكون الاختبار توليدياً.
test_record = None
with DATA_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        record = json.loads(line)
        if record.get("task_type") == "rule":
            test_record = record
            break
assert test_record is not None
test_messages = test_record["messages"][:2]
inputs = tokenizer.apply_chat_template(test_messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(answer)

In [ ]:
# 11) حفظ سجل مختصر للتشغيل
from datetime import datetime, timezone
run_log = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "data_sha256": EXPECTED_CONTENT_SHA256,
    "records": 165303,
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "peft": "MoRA",
    "mora_type": 6,
    "output_dir": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "colab_run_log.json").write_text(json.dumps(run_log, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(run_log, ensure_ascii=False, indent=2))